# Delivery Framework — Hands On

**Problem Type C:** *"Design a delivery framework that takes a customer from scoping doc to
deployed AI agent in under 2 weeks."*

This notebook builds and runs every piece of it: intake refusal, the gate sign-off engine, the
7-stage pipeline, the accelerator library, the metrics, and the negative-control (gate-failure)
demo. No LLM calls anywhere — every gate decision is deterministic, so every cell below is fast and
exactly reproducible.

Read `docs/01-theory.md` first if you have not. See `../../enterprise_rag_platform/notebooks/02-hands-on.ipynb`
for the equivalent walkthrough of Problem Type B, built the same way.

In [1]:
import sys, json, textwrap
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "delivery_framework").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

print("project root:", ROOT)

project root: d:\INTERVIEW PREPARATION\DevRev_Preparation\delivery_framework_platform


---
# Part 1 — Intake: refuse before the clock starts

§5.4 names two risk mitigations that both amount to the same thing: **refuse to start** an
engagement that was never going to be measurable. This is the delivery-framework equivalent of the
RAG project's loader refusing a document with no usable ACL — starting a 2-week clock against
something you already know can't be checked is a worse failure than never starting it.

In [2]:
from delivery_framework.pipeline import intake, ScopingRefused

# An unmeasurable request - refused outright, no Engagement object is even created.
try:
    intake(customer_name="Vague Corp", success_metrics=[],
          data_sources=["zendesk_connector"], customer_sme="u_sme_northwind")
except ScopingRefused as e:
    print("REFUSED:", e)

REFUSED: refusing to start 'Vague Corp': no measurable success metrics were agreed


In [3]:
import json
from delivery_framework.config import SETTINGS

case = json.loads((SETTINGS.data_dir / "case_study.json").read_text(encoding="utf-8"))
print(json.dumps(case, indent=2))

engagement = intake(**case)
print(f"\naccepted. stage={engagement.current_stage.label}  day={engagement.day}")

{
  "customer_name": "Northwind Logistics",
  "success_metrics": [
    "Tier-1 ticket first-response time drops below 5 minutes for the top 3 ticket categories",
    "Agent-drafted replies are sent with no human edit on at least 60% of eligible tickets by week 4"
  ],
  "data_sources": [
    "zendesk_connector",
    "confluence_connector",
    "salesforce_connector"
  ],
  "customer_sme": "u_sme_northwind"
}

accepted. stage=Scoping and qualification  day=1


---
# Part 2 — The gate sign-off engine

Every gate decision follows the same three ordered rules, deny-overrides, deliberately the same
shape as `authz.policy.decide()` in the RAG project: wrong role, missing evidence, or an
earlier-stage gate still pending are each an independent hard deny. There is no override path for
any of them in the code.

In [4]:
from delivery_framework import gates
from delivery_framework.identity import get_principal, list_principals

for p in list_principals():
    print(f"{p.user_id:<24}{p.display_name:<42}role={p.role}")

u_fda_sourav            Sourav (Forward Deployed Architect)       role=fda
u_sec_priya             Priya (Security Reviewer)                 role=security_reviewer
u_sme_northwind         Alex (Northwind Customer SME)             role=customer_sme
u_sponsor_northwind     Jordan (Northwind Exec Sponsor)           role=sponsor
u_fda_wrong_hat         Sourav, attempting a security sign-off    role=fda


In [5]:
fda = get_principal("u_fda_sourav")
sec = get_principal("u_sec_priya")
wrong_hat = get_principal("u_fda_wrong_hat")   # right person, wrong hat - role is still "fda"

# 1. Wrong role - the FDA cannot sign off the security gate, however senior.
d = gates.sign_off(engagement, "security_review_passed", wrong_hat, "trust me")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# 2. Right role, no evidence - an approval with nothing behind it is not a sign-off.
d = gates.sign_off(engagement, "security_review_passed", sec, "")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

# 3. Right role, real evidence - passes.
d = gates.sign_off(engagement, "security_review_passed", sec, "SEC-2026-0142, no blocking findings")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

DENY [wrong_role] 'fda' may not sign off 'security_review_passed'; requires one of ['security_reviewer']
DENY [no_evidence] 'security_review_passed' requires evidence, not just an approval
ALLOW [gate_signoff] 'security_review_passed' passed by security_reviewer


In [8]:
# Signing a LATER gate while an EARLIER one is still pending - denied, regardless of role/evidence.
sme = get_principal("u_sme_northwind")
d = gates.sign_off(engagement, "golden_set_signed_off", sme, "40 cases approved")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")
print("(golden_set_signed_off needs data_access_granted done first, which needs")
print(" security_review_passed done first - both are checked, not just the one before it)")

DENY [prior_gate_incomplete] cannot sign 'golden_set_signed_off' - still pending: ['data_access_granted']
(golden_set_signed_off needs data_access_granted done first, which needs
 security_review_passed done first - both are checked, not just the one before it)


---
# Part 3 — The pipeline: a stage cannot be entered while its gate is pending

`advance_stage()` only ever moves to the immediate next stage in `STAGE_ORDER` - there is no code
path that skips one, the same way retrieval in the RAG project has no code path that skips the ACL
filter.

In [9]:
from delivery_framework import engine

# Blocked: DATA_READINESS's gate (data_access_granted) hasn't passed yet.
d = engine.advance_stage(engagement)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

engagement.day = 3
engine.check_escalation_triggers(engagement)   # the automatic "escalate day 3" check
for e in engagement.escalations:
    print(f"ESCALATED on day {e.raised_on_day}: {e.reason}")

OK [advanced] entered 'Data readiness'
ESCALATED on day 3: data_access_delayed: not granted by day 3


In [10]:
from delivery_framework import accelerators

engine.record_artifact(engagement, "data_access_verification.log", sme.user_id)
d = gates.sign_off(engagement, "data_access_granted", sme,
                   "Read-only Zendesk/Confluence/Salesforce tokens verified live")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

d = engine.advance_stage(engagement, to_day=5)
print(f"{'OK' if d.allowed else 'DENY'} [{d.rule}] {d.reason}  -> now at '{engagement.current_stage.label}'")

ALLOW [gate_signoff] 'data_access_granted' passed by customer_sme
OK [advanced] entered 'Configure, do not code'  -> now at 'Configure, do not code'


In [11]:
# Configure, do not code - pull from the accelerator library before building anything custom.
accelerators.pull_or_build(engagement, "prompt_template", "support_triage_prompt")
accelerators.pull_or_build(engagement, "eval_harness", "golden_set_harness")
accelerators.pull_or_build(engagement, "guardrail_policy", "pii_redaction_policy")
accelerators.pull_or_build(engagement, "guardrail_policy", "northwind_custom_escalation_policy")  # not in the library

for p in engagement.pulls:
    print(f"{p.kind:<20}{p.name:<32}{'reused' if p.reused else 'CUSTOM-BUILT'}")

prompt_template     support_triage_prompt           reused
eval_harness        golden_set_harness              reused
guardrail_policy    pii_redaction_policy            reused
guardrail_policy    northwind_custom_escalation_policyCUSTOM-BUILT


In [12]:
engine.record_artifact(engagement, "golden_set_v1.json", sme.user_id)
gates.sign_off(engagement, "golden_set_signed_off", sme, "golden_set_v1.json reviewed, 40 cases approved")
d = engine.advance_stage(engagement, to_day=8)
print(f"{'OK' if d.allowed else 'DENY'} -> now at '{engagement.current_stage.label}'")

# Evaluate and iterate.
engine.record_eval_score(engagement, 0.83)
engine.record_artifact(engagement, "eval_baseline_report.pdf", fda.user_id)
gates.sign_off(engagement, "eval_baseline_met", fda, "eval_baseline_report.pdf - 0.83 vs 0.75 agreed baseline")
d = engine.advance_stage(engagement, to_day=10)
print(f"{'OK' if d.allowed else 'DENY'} -> now at '{engagement.current_stage.label}'")

OK -> now at 'Evaluate and iterate'
OK -> now at 'Shadow mode'


In [13]:
# Shadow mode - runs, does not act. Humans compare, some overrides expected early.
for overridden in [True, True, False, False, False]:
    engine.record_human_approval(engagement, overridden=overridden)
engine.record_artifact(engagement, "rollback_runbook.md", fda.user_id)
gates.sign_off(engagement, "rollback_tested", fda, "rollback_runbook.md - tested in staging, <2min")
d = engine.advance_stage(engagement, to_day=12)
print(f"{'OK' if d.allowed else 'DENY'} -> now at '{engagement.current_stage.label}'")

# Limited production - acts, human-approved. Override rate should be falling.
for overridden in [False, False, False, True, False, False]:
    engine.record_human_approval(engagement, overridden=overridden)
engagement.day = 14
print(f"cumulative override rate so far: {engagement.human_approval_overrides}/{engagement.human_approvals_total}")

OK -> now at 'Limited production'
cumulative override rate so far: 3/11


In [14]:
# Go/no-go - the sponsor signs off only if the ORIGINAL success metrics were actually met.
sponsor = get_principal("u_sponsor_northwind")
engine.record_artifact(engagement, "handover_runbook.md", fda.user_id)
engine.record_artifact(engagement, "dashboards_live.url", fda.user_id)

d = gates.sign_off(engagement, "success_metrics_met", sponsor,
                   "Week-2 metrics: first-response 4m12s, 63% zero-edit sends")
print(f"{'ALLOW' if d.allowed else 'DENY'} [{d.rule}] {d.reason}")

d = engine.advance_stage(engagement)
print(f"{'OK' if d.allowed else 'DENY'} -> now at '{engagement.current_stage.label}'")

d = engine.mark_deployed(engagement)
print(f"{'OK' if d.allowed else 'DENY'} {d.reason}")

ALLOW [gate_signoff] 'success_metrics_met' passed by sponsor
OK -> now at 'Go/no-go and handover'
OK success metrics met - engagement deployed


---
# Part 4 — Metrics: turning "the process works" into a number

In [15]:
from delivery_framework import metrics

for k, v in metrics.summary(engagement).items():
    print(f"{k:<28}{v}")

customer                    Northwind Logistics
stage                       Go/no-go and handover
day                         14
deployed                    True
gates_passed                1.0
time_to_first_value_days    3
eval_score_at_handover      0.83
human_approval_override_rate0.2727272727272727
week4_retention             None
accelerator_reuse_rate      0.75
open_escalations            1


**Read `accelerator_reuse_rate` carefully:** 5 of 6 pulls in Part 3 came straight from the
library; one guardrail policy was custom-built for Northwind specifically. That 83% is the actual,
numeric answer to §5.1's question — *"productised process, or bespoke heroics?"*

---
# Part 5 — Observability: the full, replayable timeline

Every gate decision, stage move, escalation, accelerator pull, and artifact - allowed or denied -
is already on `engagement.events`, because every function above logs through
`Engagement.log()` as it runs. This just renders it.

In [16]:
from delivery_framework import observability

print(observability.render_timeline(engagement))

engagement: Northwind Logistics  (day 14, stage=Go/no-go and handover)
----------------------------------------------------------------------------------------
day  1  Scoping and qualification   gate_signoff      security_review_passed    DENY[wrong_role]    'fda' may not sign off 'security_review_passed'; requires one of ['security_reviewer']
day  1  Scoping and qualification   gate_signoff      security_review_passed    DENY[no_evidence]   'security_review_passed' requires evidence, not just an approval
day  1  Scoping and qualification   gate_signoff      security_review_passed    PASS                'security_review_passed' passed by security_reviewer
day  1  Scoping and qualification   gate_signoff      golden_set_signed_off     DENY[prior_gate_incomplete]cannot sign 'golden_set_signed_off' - still pending: ['data_access_granted']
day  1  Scoping and qualification   gate_signoff      golden_set_signed_off     DENY[prior_gate_incomplete]cannot sign 'golden_set_signed_off' - still 

---
# Part 6 — What to take away

1. **A gate is a decision with a named authority behind it, not a checkbox.** Wrong role is a hard
   deny regardless of seniority - the same instinct as "the LLM is never the enforcement point" in
   the RAG project, applied to people instead of models.
2. **There is no code path that skips a stage.** `advance_stage()` only ever moves to the immediate
   next stage in order - the property is structural, not a convention someone has to remember.
3. **Refuse to start rather than start and hope.** An unmeasurable success metric or a missing SME
   is refused at intake, before day 1, the same way an unmappable document is refused at ingest in
   the RAG project.
4. **An automatic check beats a reminder.** The day-3 data-access escalation fires whether anyone
   was watching the tracker or not.
5. **"Productised vs. bespoke" is a ratio, not a claim.** `accelerator_reuse_rate` is the number
   that actually answers the question the prep doc poses in §5.1.
6. **A "no-go" is a legitimate outcome.** The framework's job is to make clear *which* gate stopped
   an engagement, not to guarantee every engagement deploys in 14 days.

Next: `../INTERVIEW_SCRIPT.md` - how to present all of this on a whiteboard in 60 minutes.